<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2011%20-%202026-05-07%20-%20Hip%C3%B3tesis%20y%20pruebas%20estad%C3%ADsticas/clase_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 11 · Hipótesis y pruebas estadísticas

**Fecha:** Jueves 7 de mayo de 2026  
**Módulo:** Módulo 3 · Modelado y estadística inferencial  

---

## Introducción

Del descubrimiento a la evidencia. Formulamos hipótesis, entendemos los tipos de error, corremos t-test, chi-cuadrado y ANOVA, e interpretamos el p-valor sin abusar de él.

## 1. Historia rápida del contraste de hipótesis

Dos escuelas dieron origen a lo que hoy llamamos *test de hipótesis*:

- **Ronald A. Fisher** (1925, *Statistical Methods for Research Workers*) — propuso el **p-valor** como medida de discrepancia entre los datos y una hipótesis nula.
- **Jerzy Neyman y Egon Pearson** (1933) — formalizaron el marco de decisión con **errores tipo I y tipo II**.

La práctica moderna mezcla ambas: reportamos p-valores (Fisher) pero decidimos con umbrales (Neyman-Pearson). Esa mezcla ha generado muchas confusiones que convienen entender.

## 2. Conceptos fundamentales

- **H₀ (hipótesis nula)**: "no hay efecto / no hay diferencia".
- **H₁ (hipótesis alternativa)**: "sí hay efecto".
- **p-valor**: probabilidad de observar datos **tan o más extremos** que los nuestros, **si H₀ fuera verdad**.
- **α (nivel de significancia)**: umbral para rechazar H₀. Convencionalmente 0.05.

**Qué NO es el p-valor**:
- No es la probabilidad de que H₀ sea cierta.
- No es la probabilidad de que los resultados se deban al azar.
- No mide el tamaño del efecto.

Un p-valor bajo significa "si no hubiera efecto, sería raro ver estos datos". Nada más.

**Errores posibles**:

| | H₀ verdadera | H₀ falsa |
|---|---|---|
| **Rechazamos H₀** | Error tipo I (α) — falso positivo | Correcto (potencia) |
| **No rechazamos** | Correcto | Error tipo II (β) — falso negativo |

## 3. Árbol de decisión · qué test usar

| Situación | Test | Supuestos |
|---|---|---|
| Comparar la **media** de 2 grupos | **t-test** (Student) | Aproximadamente normal, varianzas similares (o usar Welch) |
| Comparar **medias** de 3+ grupos | **ANOVA** (F) | Normalidad, homocedasticidad |
| Comparar 2 grupos sin asumir normalidad | **Mann-Whitney U** | No requiere normalidad |
| **Asociación** entre 2 categóricas | **Chi-cuadrado** | Frecuencias esperadas ≥ 5 |
| **Correlación** entre 2 numéricas | **Pearson** (lineal) o **Spearman** (monotónica) | — |

Si los supuestos no se cumplen, usar los tests **no paramétricos** (Mann-Whitney, Kruskal-Wallis).

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url).dropna(subset=["Age"])

# ¿La edad de hombres y mujeres difiere significativamente?
hombres = df.loc[df["Sex"] == "male", "Age"]
mujeres = df.loc[df["Sex"] == "female", "Age"]

t, p = stats.ttest_ind(hombres, mujeres, equal_var=False)
print(f"t = {t:.3f}, p = {p:.4f}")
print("Conclusión:", "rechazamos H₀" if p < 0.05 else "no podemos rechazar H₀")

## 4. Chi-cuadrado · asociación entre categóricas

El test de chi-cuadrado compara las frecuencias **observadas** con las **esperadas bajo independencia**. Si las frecuencias observadas se parecen a las esperadas → no hay asociación. Si son muy distintas → hay asociación.

In [ ]:
tab = pd.crosstab(df["Sex"], df["Survived"])
print(tab)
chi2, p, dof, esperadas = stats.chi2_contingency(tab)
print(f"\nchi2 = {chi2:.2f}, p = {p:.4e}, dof = {dof}")

## 5. ANOVA · más de 2 grupos

Si quisiéramos comparar la tarifa pagada en las 3 clases, no podemos hacer 3 t-tests (inflación de error tipo I). **ANOVA** (*Analysis of Variance*) compara las medias de varios grupos al mismo tiempo. Si sale significativo, se hace un **test post-hoc** (Tukey HSD) para identificar cuál grupo difiere de cuál.

In [ ]:
grupos = [g["Fare"].dropna() for _, g in df.groupby("Pclass")]
f, p = stats.f_oneway(*grupos)
print(f"F = {f:.2f}, p = {p:.4e}")

## 6. Tamaño del efecto · más importante que el p-valor

<img src="https://upload.wikimedia.org/wikipedia/commons/0/05/Effect_size_cohen_d.png" width="420" alt="Cohen's d">

Con una muestra grande **casi cualquier diferencia es significativa**. Por eso conviene reportar también el **tamaño del efecto**: qué tan grande es la diferencia en términos prácticos.

- **Cohen's d** para diferencias de medias: `(μ₁ - μ₂) / σ`.
  - 0.2 = pequeño, 0.5 = mediano, 0.8 = grande.
- **Cramer's V** para asociación entre categóricas.
- **r²** para regresión.

Una diferencia puede ser **estadísticamente significativa pero irrelevante** (un efecto de 0.01 con N=1M), o **no significativa pero potencialmente importante** (d=0.6 con N=10).

In [ ]:
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    s = np.sqrt(((nx - 1) * x.var() + (ny - 1) * y.var()) / (nx + ny - 2))
    return (x.mean() - y.mean()) / s

print("Cohen's d (edad hombres vs mujeres):",
      round(cohens_d(hombres, mujeres), 3))

## 7. Advertencias finales · no abusar del p-valor

- **No hacer p-hacking**: correr 20 tests y reportar el que salió significativo. Si se prueban muchas hipótesis, hay que corregir (Bonferroni, Benjamini-Hochberg).
- **No confundir significancia con causalidad**. Correlación ≠ causación.
- La American Statistical Association publicó en 2016 una declaración oficial advirtiendo contra el mal uso del p-valor. Vale la pena leerla.

---

## 🧑‍🤝‍🧑 Trabajo en grupo sobre el caso

- Formulen 2 hipótesis testeables sobre su caso.
- Corran la prueba estadística adecuada.
- Documenten la conclusión en lenguaje de negocio.

## 📦 Entregable del día

Notebook `11_hipotesis.ipynb` con 2 pruebas + conclusión.

---

## 📚 Lecturas adicionales

Para profundizar después de la clase, en [`Lecturas.md`](./Lecturas.md) encontrarás una curaduría de artículos técnicos y de negocios en español (4 por día), con copia local en la subcarpeta [`lecturas/`](./lecturas/) cuando la fuente lo permite.

### 🎬 Video recomendado

<iframe width="720" height="405" src="https://www.youtube.com/embed/Ju2Ju7q5ES0" title="ANOVA, Chi cuadrado y T de Student: cómo hacer e interpretar" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

**[ANOVA, Chi cuadrado y T de Student: cómo hacer e interpretar](https://www.youtube.com/watch?v=Ju2Ju7q5ES0)** · Todo Sobre IA

_Implementación práctica de ANOVA, t-test y chi-cuadrado con interpretación de p-valores._